# Create NMR features

Attach every NMR-derived feature block to a raw spectra dataset and save a featurized pickle for the modeling notebooks.

**Input:** a dataset with `h_nmr_peaks` / `c_nmr_peaks` peak lists (registry name or path).  
**Output:** `Datasets/<name>_featurized.pkl` with An 2014 1H descriptors, 13C bins, peak summary stats, and (optionally) NMF dictionary codes.

All feature logic lives in [`nmrlib.features`](nmrlib/features.py) so the definitions never drift between notebooks.

In [ ]:
import pandas as pd

from nmrlib import load_dataset, featurize
from nmrlib.features import feature_sets, nmf_cols

## Available datasets

Registry from [`nmrlib.data`](nmrlib/data.py). Descriptions are hardcoded labels; `found_in` is checked live against the filesystem. The column listing printed after loading (below) is the sanity check that a dataset's contents actually match its description.

In [1]:
from nmrlib import describe_datasets

describe_datasets()

,description,file,found_in
name,,,
alberts_10k,Alberts et al. 10k subset merged with qchem ta...,alberts_nmr_qchem_merged.pkl,Datasets/
alberts_10k_logp,"Alberts 10k with logP; raw spectra, pre-featur...",alberts_merged_10k_with_logp.pkl,Downloads/
alberts_10k_100kdict,Alberts 10k with NMF codes from the dictionary...,alberts_10k_100kdict_nmf_features.pkl,Downloads/
ids_nmr_1k,"IDS NMR corpus, 1k unlabeled spectra for dicti...",ids_nmr_1k.pkl,Downloads/
ids_nmr_10k,"IDS NMR corpus, 10k unlabeled spectra for dict...",ids_nmr_10k.pkl,Downloads/
ids_nmr_100k,"IDS NMR corpus, 100k unlabeled spectra for dic...",ids_nmr_100k.pkl,Downloads/
ids_1k_featurized,IDS 1k fully featurized (115-component NMF + A...,ids_1k_nmf_115_and_other.pkl,Downloads/
ids_1k_tuned_nmf,IDS 1k with tuned-NMF dictionary codes.,ids_nmr_1k_tuned_nmf_features.pkl,Downloads/
gaussian_1k,Gaussian-matched 1k set with NMR spectra.,gaussian_nmr_matched_1k.pkl,Datasets/


## Config

In [ ]:
# Raw dataset (registry short name or path to a .pkl)
dataset = "alberts_10k_logp"
out_path = "Datasets/alberts_10k_featurized.pkl"

# Optional pretrained NMF dictionary artifact to project spectra into.
# Set to None to skip NMF codes (An 2014 + 13C + stats only).
nmf_dictionary = "/Users/jacknugent/Downloads/ids_nmr_1k_dictionary_model.pkl"

## Load & inspect

In [ ]:
df = load_dataset(dataset)
df.head()
print(f"Columns ({len(df.columns)}): " + ", ".join(map(str, df.columns)))

In [ ]:
# Peek at the raw peak lists that drive every feature block
print(df[["h_nmr_peaks", "c_nmr_peaks"]].iloc[0].to_dict())

## Featurize

In [ ]:
featurized = featurize(df, dictionary_path=nmf_dictionary)
print(f"{featurized.shape[0]} rows x {featurized.shape[1]} columns")
print(f"NMF code columns: {len(nmf_cols(featurized))}")

In [ ]:
# Confirm the named feature sets the modeling notebooks expect are all present
for name, cols in feature_sets(featurized).items():
    print(f"{name:<28} {len(cols)} cols")

## Save

In [ ]:
featurized.to_pickle(out_path)
print(f"Saved -> {out_path}")